In [10]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [11]:
df = pd.read_csv("merged_data.csv.gz")

In [12]:
def feature_engineering(df):
    # AMOUNT
    df['TransactionAmt_Log'] = np.log1p(df['TransactionAmt'])
    df['Amt_decimal'] = df['TransactionAmt'] % 1
    
    card1_group = df.groupby('card1')['TransactionAmt']
    df['card1_Amt_mean'] = card1_group.transform('mean')
    df['card1_Amt_std'] = card1_group.transform('std')
    df['card1_Amt_median'] = card1_group.transform('median')
    
    df['AmountDeviationUser'] = df['TransactionAmt'] / (df['card1_Amt_mean'] + 0.001)
    df['AmountStdScore'] = (df['TransactionAmt'] - df['card1_Amt_mean']) / (df['card1_Amt_std'] + 0.001)
    df['AmountToMedianRatio'] = df['TransactionAmt'] / (df['card1_Amt_median'] + 0.001)
    
    amt_95 = df['TransactionAmt'].quantile(0.95)
    df['IsLargeTransaction'] = (df['TransactionAmt'] > amt_95).astype(int)
    df['IsSmallTestTransaction'] = (df['TransactionAmt'] < 5).astype(int)

    # TIME
    df['TransactionHour'] = (df['TransactionDT'] / 3600) % 24
    df['TransactionDayOfWeek'] = (df['TransactionDT'] / 86400) % 7
    df['IsNightTransaction'] = df['TransactionHour'].apply(lambda x: 1 if 0 <= x <= 5 else 0)
    df['IsWeekendTransaction'] = df['TransactionDayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
    df['TimeSinceLastTransaction'] = df.groupby('card1')['TransactionDT'].diff()

    df['temp_ts'] = pd.to_datetime(df['TransactionDT'], unit='s')
    df = df.sort_values('TransactionDT').reset_index(drop=True)
    temp_df = df[['card1', 'temp_ts', 'TransactionDT']].copy()
    temp_df = temp_df.set_index('temp_ts')
    
    df['TransactionVelocity1h'] = (
        temp_df.groupby('card1')['TransactionDT']
        .rolling('3600s')
        .count()
        .reset_index(level=0, drop=True)
        .values
    )
    df['TransactionVelocity24h'] = (
        temp_df.groupby('card1')['TransactionDT']
        .rolling('86400s')
        .count()
        .reset_index(level=0, drop=True)
        .values
    )
    df = df.drop('temp_ts', axis=1)
    
    # CARD
    df['CardTransactionCount'] = df.groupby('card1')['TransactionAmt'].transform('count')
    df['CardissuerFrequency'] = df['card1'].map(df['card1'].value_counts())

    df['DaysSinceRegistration'] = df['D1']
    df['AccountAgeRisk'] = 1 / (df['D1'] + 1)
    df['TimeSinceLastPurchase'] = df['D2'] 
    df['RegistrationToTransactionGap'] = df['TransactionDT'] - df['D1']
    df['D15_to_Mean_card1'] = df['D15'] / (df.groupby('card1')['D15'].transform('mean') + 0.001)

    # LOCATION
    df['AddrMismatch'] = (df['addr1'] != df['addr2']).astype(int)
    df['AddressTransactionCount'] = df.groupby('addr1')['TransactionAmt'].transform('count')
    df['CardAddressCombination'] = df.groupby(['card1', 'addr1'])['TransactionAmt'].transform('count')
    
    df['DistanceDeviation'] = df['dist1'] - df.groupby('card1')['dist1'].transform('mean')
    df['IsLongDistance'] = (df['dist1'] > 100).astype(int)
    df['DistanceRiskScore'] = (df['dist1'] - df['dist1'].min()) / (df['dist1'].max() - df['dist1'].min())
    df['card1_addr1_unique'] = df.groupby('card1')['addr1'].transform('nunique')

    # EMAIL
    df['EmailDomainMatch'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
    df['IsAnonymousEmail'] = df['P_emaildomain'].isin(['protonmail.com', 'mail.com']).astype(int)
    df['EmailDomainFrequency'] = df['P_emaildomain'].map(df['P_emaildomain'].value_counts())

    # DEVICE
    df['CardIPCount'] = df['C5']
    df['AddressDeviceCount'] = df['C7']
    df['AssociationRatio'] = (df['C1'] + df['C2']) / (df['C3'] + 0.01)
    df['TotalAssociations'] = df['C1'] + df['C2'] + df['C3']
    
    df['IsMobileDevice'] = (df['DeviceType'] == 'mobile').astype(int)
    df['null_counts'] = df.isnull().sum(axis=1)
    
    if 'id_31' in df.columns:
        df['id_31_device'] = df['id_31'].str.split(' ', expand=True)[0]
    df['Device_Freq'] = df['DeviceInfo'].map(df['DeviceInfo'].value_counts())
    df['C1_count'] = df['C1'].map(df['C1'].value_counts())

    
    v_cols = [c for c in df.columns if c.startswith('V')]
    if len(v_cols) > 0:
        pca = PCA(n_components=2)
        v_pca = pca.fit_transform(df[v_cols].fillna(-1))
        df['V_PCA_1'] = v_pca[:, 0]
        df['V_PCA_2'] = v_pca[:, 1]

    df = df.replace([np.inf, -np.inf], np.nan)
    
    return df

In [13]:
feature_engineering(df)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3812\421567868.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_Log'] = np.log1p(df['TransactionAmt'])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3812\421567868.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Amt_decimal'] = df['TransactionAmt'] % 1
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3812\421567868.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,AddressDeviceCount,AssociationRatio,TotalAssociations,IsMobileDevice,null_counts,id_31_device,Device_Freq,C1_count,V_PCA_1,V_PCA_2
0,2987000,0,86400,68.50,W,13926.0,NaN,150.0,discover,142.0,...,0.0,200.0,2.0,0,237,NaN,NaN,316791,-6602.023023,-772.102204
1,2987001,0,86401,29.00,W,2755.0,404.0,150.0,mastercard,102.0,...,0.0,200.0,2.0,0,234,NaN,NaN,316791,-6601.839626,-900.309070
2,2987002,0,86469,59.00,W,4663.0,490.0,150.0,visa,166.0,...,0.0,200.0,2.0,0,213,NaN,NaN,316791,-6601.839691,-900.309154
3,2987003,0,86499,50.00,W,18132.0,567.0,150.0,mastercard,117.0,...,0.0,700.0,7.0,0,230,NaN,NaN,105071,-6605.332700,1551.596965
4,2987004,0,86506,50.00,H,4497.0,514.0,150.0,mastercard,102.0,...,0.0,200.0,2.0,1,142,samsung,9.0,316791,163765.388993,-665.244525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550.0,NaN,150.0,visa,226.0,...,0.0,300.0,3.0,0,231,NaN,NaN,105071,-6601.918439,-871.899302
590536,3577536,0,15811049,39.50,W,10444.0,225.0,150.0,mastercard,224.0,...,0.0,200.0,2.0,0,215,NaN,NaN,316791,-6601.839691,-900.309154
590537,3577537,0,15811079,30.95,W,12037.0,595.0,150.0,mastercard,224.0,...,0.0,200.0,2.0,0,220,NaN,NaN,316791,-6601.839713,-900.309178
590538,3577538,0,15811088,117.00,W,7826.0,481.0,150.0,mastercard,224.0,...,0.0,200.0,2.0,0,211,NaN,NaN,316791,-6605.305878,1150.609027
